# M02 — Ingesta y preparación

[← Anterior](../M01-fundamentos-entorno/02-lab-sesion-spark.ipynb) · [Siguiente →](02-lab-ingesta-csv-json.ipynb)

Hasta ahora las filas las inventábamos. A partir de aquí leemos **ficheros reales** de NovaShop (`data/raw/`).

La pregunta de este módulo no es “¿cuál es el API de `read.csv`?”. Es: **quién decide cómo se llama cada columna y de qué tipo es**. Eso lo puedes dejar en manos de Spark (mira el fichero y adivina) o lo puedes **escribir tú** (un contrato). En exploración vale lo primero. En un pipeline que mañana vuelve a correr, lo segundo.

Labs después: cargar → tipar → limpiar.

Ejecuta las celdas **aquí**, en este mismo fichero (clase, juntos). Va **montado**: explicación + código + lo que tienes que ver. Lo que construyes tú está en el **lab**.

Kernel: **Python (NovaShop)**.


## Arranque

La primera celda **no es Spark todavía**: busca la raíz del repo (aunque este notebook no esté en la carpeta de arriba) y deja `RAW`, `STAGING` y `CURATED` listos. La segunda pide una `SparkSession` en `local[*]` (todos los cores de esta máquina; no hay clúster).

Al ejecutar: rutas impresas y una versión `3.5.x` con master `local[*]`.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
# getOrCreate: si ya hay sesión en este kernel, la reusa (mismo puerto 4040)
spark = get_spark('novashop-clase-m02')
print(spark.version, spark.sparkContext.master)


## Qué hace Spark cuando lee un CSV “a pelo”

Un CSV es texto. No lleva tipos: no dice “esto es un timestamp” ni “esto es un decimal”. Si no le pasas un contrato, Spark **mira una muestra** del fichero y decide nombres y tipos. Eso es adivinar a partir de los datos (en la jerga: *inferir*).

En CSV, esa adivinación es especialmente vaga: casi todo acaba en **string**, aunque la columna se llame `OrderDate` o parezca un número. No es un fallo. Es “no me has dicho el tipo, no me la juego”.

Vamos a leer `orders.csv` solo con `header=True` (la primera línea son nombres). Al ejecutar verás:

- `count` → **800** (si sale 801, has contado la cabecera: falta el `header`).
- `printSchema()` → todas las columnas `string`, con nombres camelCase (`OrderId`, `OrderDate`…).
- `show(3)` → tres filas crudas, tal cual el fichero.


In [ ]:
# Sin .schema(...): Spark decide nombres (por la cabecera) y tipos (casi todo string).
orders_txt = spark.read.option("header", True).csv(str(RAW / "orders.csv"))
print("filas", orders_txt.count())  # 800
orders_txt.printSchema()  # espera string, string, string...
orders_txt.show(3, truncate=False)


## Dos JSON que no se leen igual

NovaShop trae dos JSON distintos. El método se llama igual (`.json`); el fichero no.

- `products.json` es **un solo documento**: un array `[ {...}, {...} ]`. Si Spark lee *línea a línea*, cada línea es un trozo de JSON roto y marca `_corrupt_record`. Por eso `multiLine=True`: “este fichero es un JSON entero, no un JSON por línea”.
- `events.jsonl` es **una línea = un objeto**. Ahí el valor por defecto va bien.

Al ejecutar: `products` **60**, `events` **2500**. En el schema de productos verás camelCase (`productId`, `listPrice`). Si `products` te sale ~362, falta el `multiLine`.


In [ ]:
# Array JSON (un documento) vs JSONL (un objeto por línea)
products = spark.read.option("multiLine", True).json(str(RAW / "products.json"))
events = spark.read.json(str(RAW / "events.jsonl"))
print("products", products.count(), "events", events.count())
products.printSchema()


## Tú escribes el contrato (y luego parseas las fechas)

Dejar que Spark adivine está bien para **mirar**. Para **producir**, escribes el contrato: lista de columnas, tipo de cada una, y si admite nulos. En el API eso es un `StructType` de `StructField`. No borra filas: solo dice “lee estas columnas así”.

En NovaShop el fichero mezcla dos textos de fecha: la mayoría `yyyy-MM-dd HH:mm:ss` y tres en `dd/MM/yyyy`. Si conviertes con **un** solo formato, esas tres se vuelven nulas. No desaparece el pedido: desaparece la fecha.

Qué vas a ver en la siguiente celda, en este orden:

1. Las tres fechas con `/` (suciedad deliberada).
2. Un contrato que **todavía deja `OrderDate` en string** (el texto crudo). Renombramos a `snake_case`.
3. `coalesce` de dos `to_timestamp`: “prueba ISO; si falla, prueba día/mes/año”.
4. `printSchema()` con `order_ts` ya en `timestamp`.
5. Nulos de fecha → **0**. El `count` sigue siendo **800**. Tipar no limpia claves ni tira filas: eso es el lab de calidad.


In [ ]:
from pyspark.sql.functions import col, coalesce, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType

# 1) Suciedad que el adivinador no te cuenta: tres fechas con barra
print("fechas raras (dd/mm/yyyy):")
orders_txt.where(col("OrderDate").contains("/")).select("OrderId", "OrderDate").show()

# 2) Contrato: nombres del FICHERO y tipos de lectura (aún texto).
#    True = la columna puede ser nula.
schema = StructType([
    StructField("OrderId", StringType(), True),
    StructField("CustomerId", StringType(), True),
    StructField("OrderDate", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("Channel", StringType(), True),
])

orders = (
    spark.read.option("header", True)
    .schema(schema)  # ya no adivina: usa esta lista
    .csv(str(RAW / "orders.csv"))
    .withColumnRenamed("OrderId", "order_id")
    .withColumnRenamed("CustomerId", "customer_id")
    .withColumnRenamed("OrderDate", "order_ts_raw")
    .withColumnRenamed("Status", "status")
    .withColumnRenamed("Channel", "channel")
    # 3) Dos formatos; coalesce se queda con el primero que no sea nulo
    .withColumn(
        "order_ts",
        coalesce(
            to_timestamp(col("order_ts_raw"), "yyyy-MM-dd HH:mm:ss"),
            to_timestamp(col("order_ts_raw"), "dd/MM/yyyy"),
        ),
    )
    .drop("order_ts_raw")
)
orders.printSchema()
print("nulos de fecha", orders.where(col("order_ts").isNull()).count())  # 0
print("count sigue siendo", orders.count())  # 800: tipar ≠ filtrar


En los labs vas a repetir esta idea con líneas (enteros y decimales) y con eventos. El contrato lo escribes tú; Spark no tiene que “acertar” cada mañana.

**Siguiente:** [lab de ingesta](02-lab-ingesta-csv-json.ipynb) — creas tu notebook y cargas las cuatro fuentes.
